# Kraken HTR on the cluster — a hands-on demo

Train a text-recognition (HTR) model for **Latin stone inscriptions** with the
[Kraken](https://kraken.re) engine — a small, self-contained use case to try the
cluster's JupyterHub.

You get a ready-to-run dataset and **four training recipes** to compare:
**real only · synthetic pretrain→finetune · mixed · weighted mixed**.

- **Data:** 50 annotated inscriptions from the *Epigraphic Database Heidelberg*
  (EDH, **CC BY-SA 4.0**) + 500 synthetic stone images.
- **Validation uses real data only** — synthetic images are for training, never scored on.
- **Plain & simple:** one train/val split, no cross-validation.

## 0 · Setup
Install Kraken (if your env doesn't have it) and fetch **CATMuS-Print Large** — a strong
printed-text base model we fine-tune.

In [ ]:
# Ensure Kraken is installed in THIS kernel's environment (provides the `ketos`/`kraken` CLIs).
import sys, os, glob, re, shutil, site, sysconfig
from pathlib import Path

if shutil.which("ketos") is None:
    print("installing kraken (one-time) ...")
    !{sys.executable} -m pip install -q kraken
# Put the console-scripts dirs on PATH for the `!` shell cells. On a read-only base env
# pip installs with --user, so the scripts land in ~/.local/bin, which is NOT on PATH.
for _bin in (os.path.join(site.getuserbase(), "bin"), sysconfig.get_path("scripts")):
    if _bin and _bin not in os.environ.get("PATH", "").split(os.pathsep):
        os.environ["PATH"] = _bin + os.pathsep + os.environ.get("PATH", "")
assert shutil.which("ketos"), (
    "kraken still not on PATH. If pip failed (read-only base env), make a writable venv:\n"
    "  python -m venv ~/kraken-venv && ~/kraken-venv/bin/pip install kraken ipykernel\n"
    "  ~/kraken-venv/bin/python -m ipykernel install --user --name kraken-venv\n"
    "then switch the notebook kernel to 'kraken-venv' (top-right) and re-run.")
print("kraken:", shutil.which("ketos"))

# fetch the CATMuS-Print Large base model (downloads once into ~/.local/share/htrmopo)
!kraken get 10.5281/zenodo.10592716
BASE = glob.glob(os.path.expanduser("~/.local/share/htrmopo/**/*fondue-large*.mlmodel"), recursive=True)[0]
print("base model:", BASE)
print("✅ ready")

## 1 · The data
Each example is an image + a PAGE-XML with the diplomatic transcription in
*scriptio continua* (the Latin convention: no spaces between words).

In [ ]:
from PIL import Image
from IPython.display import display
def gt(xml):
    return "  ".join(re.findall(r"<Unicode>([^<]*)</Unicode>", Path(xml).read_text(encoding="utf-8"))[:4])

real  = sorted(glob.glob("data/real/page/*.xml"))
synth = sorted(glob.glob("data/synth/page/*.xml"))
print(f"real annotated: {len(real)}    synthetic: {len(synth)}")
print("real  example:", gt(real[0]))
print("synth example:", gt(synth[0]))
display(Image.open("data/real/images/" + Path(real[0]).stem + ".jpg"))   # a real inscription
print("✅ ready")

## 2 · Train / val split + helpers
40 real → training, 10 real → validation. **Synthetic data is training-only — never in
validation**, so every score is measured on real inscriptions.

In [ ]:
for d in ("lists", "arrows", "mf", "models"): os.makedirs(d, exist_ok=True)
train_real, val_real = real[:40], real[40:50]
Path("lists/train_real.txt").write_text("\n".join(train_real))
Path("lists/val_real.txt").write_text("\n".join(val_real))
Path("lists/synth.txt").write_text("\n".join(synth))

def mf(p, arrows): Path(p).write_text("\n".join(arrows) + "\n")
mf("mf/val.mf",      ["arrows/val_real.arrow"])
mf("mf/real.mf",     ["arrows/train_real.arrow"])
mf("mf/synth.mf",    ["arrows/synth.arrow"])
mf("mf/mixed.mf",    ["arrows/train_real.arrow", "arrows/synth.arrow"])
mf("mf/weighted.mf", ["arrows/train_real.arrow"] * 5 + ["arrows/synth.arrow"])

def cer(model_dir):
    "final validation letter-CER from the best_<acc>.safetensors filename (acc = char accuracy)"
    best = sorted(glob.glob(f"{model_dir}/best_*.safetensors"))
    if not best: return None
    return round((1 - float(re.search(r"best_([0-9]+\.[0-9]+)", best[-1]).group(1))) * 100, 1)

TRAIN = "-B 32 --resize new -q early --min-epochs 3 --lag 3 -N 30"   # small + quick for the demo
print(f"train_real={len(train_real)}   val_real={len(val_real)} (REAL only)   synth(train)={len(synth)}")
print("✅ ready")

### Compile the binary datasets
Kraken trains fastest from a precompiled dataset — one per source.

In [ ]:
!ketos compile -f page -o arrows/train_real.arrow $(cat lists/train_real.txt)
!ketos compile -f page -o arrows/val_real.arrow   $(cat lists/val_real.txt)
!ketos compile -f page -o arrows/synth.arrow      $(cat lists/synth.txt)
print("✅ ready")

## 3 · Four recipes
Each fine-tunes the CATMuS base, validates on the 10 held-out **real** inscriptions, and
reports **letter-CER** (lower = better).

### A — real only
The honest baseline with scarce data.

In [ ]:
!ketos train -f binary -t mf/real.mf -e mf/val.mf -i {BASE} -o models/real {TRAIN}
print("\n>>> real-only  CER =", cer("models/real"), "%")
print("✅ ready")

### B — synthetic pretrain → finetune
Pretrain on the 500 synthetic stones, then fine-tune on the real data. (Kraken saves
checkpoints; we convert the best one to a `.mlmodel` so it can be the fine-tuning base.)

In [ ]:
# 1) pretrain on synthetic
!ketos train -f binary -t mf/synth.mf -e mf/val.mf -i {BASE} -o models/pretrain {TRAIN}
# 2) pick the best checkpoint (robust to naming) and convert it to a .mlmodel base
scored = [(float(m.group(1)), p) for p in glob.glob("models/pretrain/*.ckpt")
          if (m := re.search(r"([0-9]+\.[0-9]+)\.ckpt$", p))]
if scored:
    src = max(scored)[1]                                              # checkpoint with the highest val acc
else:
    cks = sorted(glob.glob("models/pretrain/*.ckpt"), key=os.path.getmtime)
    src = cks[-1] if cks else sorted(glob.glob("models/pretrain/best_*.safetensors"))[-1]
print("converting:", src)
!rm -f models/pretrain/model.mlmodel
!ketos convert --weights-format coreml -o models/pretrain/model.mlmodel {src}
# 3) finetune on real
!ketos train -f binary -t mf/real.mf -e mf/val.mf -i models/pretrain/model.mlmodel -o models/pretrain_ft {TRAIN}
print("\n>>> pretrain->finetune  CER =", cer("models/pretrain_ft"), "%")
print("✅ ready")

### C — mixed
Train on real + synthetic together in one run.

In [ ]:
!ketos train -f binary -t mf/mixed.mf -e mf/val.mf -i {BASE} -o models/mixed {TRAIN}
print("\n>>> mixed  CER =", cer("models/mixed"), "%")
print("✅ ready")

### D — weighted mixed
Same as mixed, but the scarce real data is **oversampled ×5** so the 500 synthetic don't
swamp the 40 real.

In [ ]:
!ketos train -f binary -t mf/weighted.mf -e mf/val.mf -i {BASE} -o models/weighted {TRAIN}
print("\n>>> weighted-mixed  CER =", cer("models/weighted"), "%")
print("✅ ready")

## 4 · Compare

In [ ]:
res = {
    "real only":               cer("models/real"),
    "pretrain -> finetune":    cer("models/pretrain_ft"),
    "mixed":                   cer("models/mixed"),
    "weighted mixed (real x5)":cer("models/weighted"),
}
print(f"{'recipe':28s} val letter-CER")
print("-" * 45)
for k, v in sorted(res.items(), key=lambda kv: (kv[1] is None, kv[1])):
    print(f"{k:28s} {v}%")
print("\n(lower is better; validated on the 10 held-out REAL inscriptions)")
print("✅ ready")

---
**Data licence:** the 50 real inscriptions are © *Epigraphic Database Heidelberg*,
reused under **CC BY-SA 4.0** — see [`data/LICENSE.txt`](data/LICENSE.txt). The 500
synthetic images are released under the same licence.